In [1]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device use : {device}")

device use : cuda


In [3]:
df = pd.read_csv("/content/fmnist_small.csv")
df.sample(5)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
1282,8,0,0,0,0,0,0,0,0,0,...,117.0,121.0,92.0,97.0,96.0,101.0,79.0,80.0,52.0,0.0
123,6,0,0,0,0,0,0,0,0,1,...,45.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3888,0,0,0,0,0,0,0,0,0,7,...,40.0,44.0,68.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2853,6,0,0,0,0,0,0,0,0,3,...,0.0,1.0,0.0,65.0,95.0,57.0,77.0,0.0,0.0,0.0
4075,6,0,0,0,0,0,1,0,0,0,...,79.0,94.0,0.0,0.0,150.0,154.0,77.0,13.0,0.0,0.0


In [4]:
x = df.drop(columns=['label']).values
y = df['label'].values

In [5]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2, random_state = 42)

In [7]:
from torchvision.transforms import transforms

custom_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
    ]
)

In [8]:
from PIL import Image
import numpy as np
class sampledata(Dataset):
  def __init__(self,features,labels,transform):
    self.features = features
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    #resize images into(28,28)

    image = self.features[index].reshape(28,28)
    #change datatype into np.uint8

    image = image.astype(np.uint8)
    #convert back and white to color 3 channel form
    image = np.stack([image]*3, axis = -1) # axis=-1 mean (H,W,C) -> (C,H,W)

    #convert array to PIL image
    image = Image.fromarray(image)

    #apply transforms
    image = self.transform(image)

    return image, torch.tensor(self.labels[index], dtype = torch.long)


In [9]:
train_dataset = sampledata(x_train,y_train,transform = custom_transform)
test_dataset = sampledata(x_test,y_test,transform = custom_transform)

In [10]:
train_loader = DataLoader(train_dataset,batch_size = 32,shuffle = True,pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size = 32,shuffle = False,pin_memory = True)

In [12]:
import torchvision.models as models

vgg16 = models.vgg16(pretrained = True)

vgg16

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 148MB/s]


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [13]:
#frezze the feature extraction layers
for param in vgg16.features.parameters():
  param.requires_grad = False

In [14]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088,1024),
    nn.ReLU(),
    nn.Dropout(0.5),

    nn.Linear(1024,512),
    nn.ReLU(),
    nn.Dropout(0.5),

    nn.Linear(512,10)
)

In [15]:
vgg16 = vgg16.to(device)

In [16]:
learning_rate = 0.0001
epochs = 10

In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16.classifier.parameters(), lr = learning_rate)

In [22]:
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch,labels in train_loader:
    batch = batch.to(device)
    labels = labels.to(device)
    output = vgg16(batch)
    loss = criterion(output,labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_epoch_loss += loss.item()
  print(f"epoch num {epoch+1}, loss : {loss.item()}")

epoch num 1, loss : 0.07629422098398209
epoch num 2, loss : 1.4235050678253174
epoch num 3, loss : 0.0368078239262104
epoch num 4, loss : 0.025213519111275673
epoch num 5, loss : 0.2576344907283783
epoch num 6, loss : 0.22790634632110596
epoch num 7, loss : 0.0007105885888449848
epoch num 8, loss : 0.016035156324505806
epoch num 9, loss : 0.0004005632072221488
epoch num 10, loss : 8.16580268292455e-06


In [23]:
vgg16.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [24]:
# evaluation on test data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = vgg16(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)


/tmp/ipykernel_4105/3108301102.py:18: RuntimeWarning: invalid value encountered in cast
  image = image.astype(np.uint8)


0.8835978835978836


In [26]:
# evaluation on training data
total = 0
correct = 0

with torch.no_grad():

  for features, labels in train_loader:

    # move data to gpu
    features, labels = features.to(device), labels.to(device)

    outputs = vgg16(features)

    _, predicted = torch.max(outputs, 1)

    total = total + labels.shape[0]

    correct = correct + (predicted == labels).sum().item()

print(correct/total)

1.0
